# 10.7 Conditions and loop control

In [1]:
# install dependencies
%pip install -q amplpy pandas

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

The `for` loops of [Section 10.6](10_6_iterating_over_a_set_the_for_statement.ipynb) make one pass for each member of a collection that is known in advance. This section describes how to iterate as long as some logical condition is satisfied, how to perform commands only when a condition holds, and how to terminate a loop or a pass through it early. We continue with the multi-period production model `steelT` of [Section 4.2](../04/4_2_a_multiperiod_production_model.ipynb):

In [2]:
%%writefile steelT.mod

set PROD;     # products
param T > 0;  # number of weeks

param rate {PROD} > 0;          # tons per hour produced
param inv0 {PROD} >= 0;         # initial inventory
param avail {1..T} >= 0;        # hours available in week
param market {PROD,1..T} >= 0;  # limit on tons sold in week

param prodcost {PROD} >= 0;     # cost per ton produced
param invcost {PROD} >= 0;      # carrying cost/ton of inventory
param revenue {PROD,1..T} >= 0; # revenue per ton sold

var Make {PROD,1..T} >= 0;      # tons produced
var Inv {PROD,0..T} >= 0;       # tons inventoried
var Sell {p in PROD, t in 1..T} >= 0, <= market[p,t]; # tons sold

# Total revenue less costs in all weeks
maximize Total_Profit:
   sum {p in PROD, t in 1..T} (revenue[p,t]*Sell[p,t] -
      prodcost[p]*Make[p,t] - invcost[p]*Inv[p,t]);

# Total of hours used by all products
# may not exceed hours available, in each week
subject to Time {t in 1..T}:
   sum {p in PROD} (1/rate[p]) * Make[p,t] <= avail[t];

# Initial inventory must equal given value
subject to Init_Inv {p in PROD}:  Inv[p,0] = inv0[p];

# Tons produced and taken from inventory
# must equal tons sold and put into inventory
subject to Balance {p in PROD, t in 1..T}:
   Make[p,t] + Inv[p,t-1] = Sell[p,t] + Inv[p,t];

Writing steelT.mod


In [3]:
import pandas as pd

steelT_prod = pd.DataFrame(
    [
        ["bands", 200, 10, 10, 2.5],
        ["coils", 140,  0, 11, 3],
    ],
    columns=["PROD", "rate", "inv0", "prodcost", "invcost"],
).set_index("PROD")

steelT_week = pd.DataFrame(
    [
        ["bands", 1, 25, 6000],
        ["bands", 2, 26, 6000],
        ["bands", 3, 27, 4000],
        ["bands", 4, 27, 6500],
        ["coils", 1, 30, 4000],
        ["coils", 2, 35, 2500],
        ["coils", 3, 37, 3500],
        ["coils", 4, 39, 4200],
    ],
    columns=["PROD", "T", "revenue", "market"],
).set_index(["PROD", "T"])

steelT_avail = {1: 40, 2: 40, 3: 32, 4: 40}

In [4]:
def load_steelT(ampl):
    """Read the steelT model and assign its data."""
    ampl.read("steelT.mod")
    ampl.param["T"] = 4
    ampl.param["avail"] = steelT_avail
    ampl.set_data(steelT_prod, "PROD")
    ampl.set_data(steelT_week)


load_steelT(ampl)

## 10.7.1 Iterating subject to a condition: the `while` statement

A second kind of looping construct, the `while` statement, continues iterating as long as some logical condition is satisfied.

Returning to the sensitivity analysis example, we wish to take advantage of a property of the dual value on the constraint `Time[3]`: the additional profit that can be realized from each extra hour added to `avail[3]` is at most `Time[3].dual`. When `avail[3]` is made sufficiently large, so that there is more third-week capacity than can be used, the associated dual value falls to zero and further increases in `avail[3]` have no effect on the optimal solution.

We can specify that looping should stop once the dual value falls to zero, by writing a `while` statement:

```python
while dual > 0:
    ...
```

The loop body, here indicated by `...`, is the indented block beneath the `while` line. The condition is tested before every pass, and passes through the loop continue as long as it is true; if the condition is false before the first pass, then the loop body is never executed. Python has no form of the loop that tests its condition after every pass. When the body must be executed at least once, as here, where the dual value is not known until a solve has been executed in the first pass, the usual Python idiom is a loop `while True:` that repeats indefinitely, terminated from within the body by the `break` statement described below:

```python
while True:
    ...
    if dual == 0:
        break
```

A complete script using this form is shown in [Figure 10-3](#fig-10-3). At the beginning of the script, we don't know how many passes the loop will make. Thus we cannot determine the values of `avail[3]` in advance as we did in [Figure 10-1](10_6_iterating_over_a_set_the_for_statement.ipynb#fig-10-1). Instead, we start with empty dictionaries, and add an entry for the current value of `avail[3]` after each solve:

In [5]:
import pandas as pd

avail3_obj = {}
avail3_dual = {}
avail3_step = 5

while True:
    ampl.solve(solver="highs", verbose=False)
    assert ampl.solve_result == "solved", ampl.solve_result

    a = ampl.param["avail"][3]
    dual = ampl.con["Time"][3].dual()
    avail3_obj[a] = ampl.obj["Total_Profit"].value()
    avail3_dual[a] = dual

    ampl.param["avail"][3] = a + avail3_step
    if dual <= 1e-7:   # the dual value has fallen to zero
        break

pd.DataFrame({"avail3_obj": avail3_obj, "avail3_dual": avail3_dual})

,avail3_obj,avail3_dual
32,515033.0,3400.0
37,532033.0,3400.0
42,549033.0,3400.0
47,565193.0,2980.0
52,580093.0,2980.0
57,594993.0,2980.0
62,609893.0,2980.0
67,624793.0,2980.0
72,626283.0,0.0


<a id='fig-10-3'></a>**[Figure 10-3](#fig-10-3):** Script for recording sensitivity.

Notice that the loop ends when the dual value is at most `1e-7`, rather than when it is exactly zero. Because numbers in the computer are represented with a limited number of bits of precision, a solver may return values that differ very slightly from the solution that would be computed using exact arithmetic. Ordinarily you don't see this, because the `display` command rounds values to six significant digits by default. Setting `display_precision` to 0 drops this rounding:

In [6]:
ampl.option["display_precision"] = 0
ampl.display("Make")

Make :=
bands 1   5990
bands 2   6000
bands 3   4000
bands 4   6500
coils 1   1407
coils 2   1400
coils 3   6650
coils 4   1050
;



For this problem HiGHS happens to return exact values, but that cannot be relied upon in general; with other solvers, or other problems, a value that should be 1400 may well come back as `1399.9999999999998`. The same effect is easily seen in Python itself:

In [7]:
0.1 * 3, 0.1 * 3 == 0.3

(0.30000000000000004, False)

These seemingly tiny differences can have undesirable effects whenever a script makes a comparison that uses values returned by the solver. A test such as `make["coils", 2] >= 1400` could turn out false when the rounded display would lead you to believe that it is true.

You can avoid this kind of surprise by writing arithmetic tests more carefully: instead of `dual == 0`, for instance, you might test `dual <= 1e-7`, as in [Figure 10-3](#fig-10-3), and for comparing two computed values Python provides `math.isclose()`. Alternatively, you can round all solution values that are returned by the solver, so that numbers that are supposed to be equal really do come out equal. The setting

In [8]:
ampl.option["display_precision"] = 6
ampl.option["solution_precision"] = 10

states that solution values are to be rounded to 10 significant digits. This and related rounding options are discussed and illustrated in [Section 12.3](../12/12_3_numeric_options_for_display.ipynb).

## 10.7.2 Testing a condition: the `if` statement

In [Section 7.3](../07/7_3_logical_and_conditional_expressions.ipynb) we described AMPL's conditional (if-then-else) expression, which produces an arithmetic or set value that can be used in any expression context; Python's conditional expression, `x if condition else y`, works the same way. The `if` statement uses a condition to control the execution of statements or groups of statements.

In the simplest case, the `if` statement evaluates a condition and takes a specified action if the condition is true. After solving the original problem again, we retrieve the values of `Make` and test one of them:

In [9]:
ampl.reset()
load_steelT(ampl)
ampl.solve(solver="highs", verbose=False)
assert ampl.solve_result == "solved", ampl.solve_result

make = ampl.var["Make"].to_dict()

if make["coils", 2] < 1500:
    print("under 1500")

under 1500


The action may also be a series of statements, all indented beneath the `if` as in the `for` and `while` statements, and an optional `else` specifies an alternative action:

In [10]:
if make["coils", 2] < 1500:
    print("Fewer than 1500 coils in week 2.")
    ampl.param["market"]["coils", 2] = ampl.param["market"]["coils", 2] * 1.1
else:
    print("At least 1500 coils in week 2.")
    ampl.param["market"]["coils", 2] = ampl.param["market"]["coils", 2] * 0.9

ampl.param["market"]["coils", 2]

Fewer than 1500 coils in week 2.


2750

Python executes these statements by first evaluating the logical expression following `if`. If the expression is true, the statements following it are executed. If the expression is false, the statements following `else`, if any, are executed. Because the grouping of statements is shown by their indentation, there is never any doubt about which `if` an `else` belongs to.

The `if` statement is most useful for regulating the flow of control in scripts. In [Figure 10-2](10_6_iterating_over_a_set_the_for_statement.ipynb#fig-10-2), we could suppress any occurrences of 100% by placing the statement that prints `Sell[p,t]/market[p,t]` inside an `if`:

```python
if sell[p, t] < market[p, t]:
    print(f"{100 * sell[p, t] / market[p, t]:7.1f}%", end="")
else:
    print("    --- ", end="")
```

The statement following `if` or `else` can itself be an `if` statement. In the formatting example, we could handle both 0% and 100% specially by nesting a second `if`, or more clearly by using `elif` (short for "else if"):

In [11]:
ampl.reset()
load_steelT(ampl)
ampl.solve(solver="highs", verbose=False)
assert ampl.solve_result == "solved", ampl.solve_result

sell = ampl.var["Sell"].to_dict()
market = ampl.param["market"].to_dict()
T = ampl.param["T"].value()
PROD = ampl.set["PROD"].to_list()

print("\nSALES", end="")
for p in PROD:
    print(f"{p:>14}   ", end="")
print()
for t in range(1, T + 1):
    print(f"week {t}", end="")
    for p in PROD:
        print(f"{sell[p, t]:9.0f}", end="")
        if sell[p, t] == 0:
            print("        ", end="")
        elif sell[p, t] < market[p, t]:
            print(f"{100 * sell[p, t] / market[p, t]:7.1f}%", end="")
        else:
            print("    --- ", end="")
    print()


SALES         bands            coils   
week 1     6000    ---       307    7.7%
week 2     6000    ---      2500    --- 
week 3     1400   35.0%     3500    --- 
week 4     2000   30.8%     4200    --- 


In the script of [Figure 10-3](#fig-10-3), we can use an `if` statement inside the loop to test whether the dual value has changed since the previous pass through the loop, as shown in the script of [Figure 10-4](#fig-10-4). This loop starts from `avail[3]` equal to 1 and increases it in steps of 1, but creates a table that has exactly one entry for each different dual value discovered. The variable `previous_dual` starts out as infinity (`math.inf`), so that the condition of the `while` statement is true for the first pass:

In [12]:
import math

avail3_obj = {}
avail3_dual = {}
ampl.param["avail"][3] = 1
avail3_step = 1
previous_dual = math.inf

while previous_dual > 1e-7:
    ampl.solve(solver="highs", verbose=False)
    assert ampl.solve_result == "solved", ampl.solve_result

    a = ampl.param["avail"][3]
    dual = ampl.con["Time"][3].dual()
    if dual < previous_dual:
        avail3_obj[a] = ampl.obj["Total_Profit"].value()
        avail3_dual[a] = dual
        previous_dual = dual

    ampl.param["avail"][3] = a + avail3_step

pd.DataFrame({"avail3_obj": avail3_obj, "avail3_dual": avail3_dual})

,avail3_obj,avail3_dual
1,404616.142864,3620.0
23,484233.000000,3500.0
26,494633.000000,3400.0
46,562213.000000,2980.0
68,626283.000000,0.0


<a id='fig-10-4'></a>**[Figure 10-4](#fig-10-4):** Testing conditions within a loop.

## 10.7.3 Terminating a loop: `break` and `continue`

Two other statements work with looping statements to make some scripts easier to write. The `continue` statement terminates the current pass through a `for` or `while` loop; all further statements in the current pass are skipped, and execution continues with the test that controls the start of the next pass (if any). The `break` statement completely terminates a `for` or `while` loop, sending control immediately to the statement following the end of the loop.

As an example of both these statements, [Figure 10-5](#fig-10-5) exhibits another way of writing the loop from [Figure 10-4](#fig-10-4), so that a table entry is made only when there is a change in the dual value associated with `avail[3]`. After solving, we test to see if the new dual value is equal to the previous one:

```python
if math.isclose(dual, previous_dual):
    continue
```

If it is, there is nothing to be done for this value of `avail[3]`, and the `continue` statement jumps to the end of the current pass; execution resumes with the next pass, starting at the beginning of the loop.

After adding an entry to the table, we test to see if the dual value has fallen to zero:

```python
if dual <= 1e-7:
    break
```

If it has, the loop is done and the `break` statement jumps out; execution passes to the statement that follows the loop in the script. Since the loop in this example is `while True`, it relies on the `break` statement for termination.

In [13]:
avail3_obj = {}
avail3_dual = {}
ampl.param["avail"][3] = 0
previous_dual = math.inf

while True:
    ampl.param["avail"][3] = ampl.param["avail"][3] + 1
    ampl.solve(solver="highs", verbose=False)
    assert ampl.solve_result == "solved", ampl.solve_result

    a = ampl.param["avail"][3]
    dual = ampl.con["Time"][3].dual()
    if math.isclose(dual, previous_dual):
        continue
    avail3_obj[a] = ampl.obj["Total_Profit"].value()
    avail3_dual[a] = dual
    if dual <= 1e-7:
        break
    previous_dual = dual

pd.DataFrame({"avail3_obj": avail3_obj, "avail3_dual": avail3_dual})

,avail3_obj,avail3_dual
1,404616.142864,3620.0
23,484233.000000,3500.0
26,494633.000000,3400.0
46,562213.000000,2980.0
68,626283.000000,0.0


<a id='fig-10-5'></a>**[Figure 10-5](#fig-10-5):** Using `break` and `continue` in a loop.

When a `break` or `continue` lies within a nested loop, it applies only to the innermost loop. This convention generally has the desired effect. As an example, consider how we could expand [Figure 10-5](#fig-10-5) to perform a separate sensitivity analysis on each `avail[t]`. The `while` loop would be nested in a `for t in range(1, T + 1)` loop, but the `continue` and `break` statements would apply to the inner `while` as before.

There do exist situations in which the logic of a script requires breaking out of multiple loops. In the script of [Figure 10-5](#fig-10-5), for instance, we can imagine that instead of stopping when `Time[3].dual` is zero, we want to stop when `Time[t].dual` falls below 2700 for any `t`. It might seem that one way to implement this criterion is:

```python
for t in range(1, T + 1):
    if ampl.con["Time"][t].dual() < 2700:
        break
```

This statement does not have the desired effect, however, because `break` applies only to the inner `for` loop that contains it, rather than to the outer `while` loop as we desire. Python has no way of naming a loop in a `break` statement, but in this case the inner loop is not needed at all: the built-in function `any()` tests whether a condition holds for any member of a collection, so that the test can be written as a single condition of the outer loop:

In [14]:
avail3_obj = {}
avail3_dual = {}
ampl.param["avail"][3] = 0
previous_dual = math.inf

while True:
    ampl.param["avail"][3] = ampl.param["avail"][3] + 1
    ampl.solve(solver="highs", verbose=False)
    assert ampl.solve_result == "solved", ampl.solve_result

    a = ampl.param["avail"][3]
    dual = ampl.con["Time"][3].dual()
    if math.isclose(dual, previous_dual):
        continue
    avail3_obj[a] = ampl.obj["Total_Profit"].value()
    avail3_dual[a] = dual
    if any(ampl.con["Time"][t].dual() < 2700 for t in range(1, T + 1)):
        break
    previous_dual = dual

pd.DataFrame({"avail3_obj": avail3_obj, "avail3_dual": avail3_dual})

,avail3_obj,avail3_dual
1,404616.142864,3620.0
23,484233.000000,3500.0


When the inner loop is more complicated, the usual Python technique is to place the loops in a function, and to leave all of them at once with a `return` statement; functions were introduced in [Section 10.5](10_5_running_scripts.ipynb).